In [1]:
"""
Diagnose why valid pixel counts differ between GT and classified TIF
"""
import numpy as np
import rasterio

GT_PATH  = "/beluga/Hackathon15_AlphaEarth_Alberta/Hackathon15_AlphaEarth_Alberta/GroundTruth_Landsat_Canada/landcover-2020-classification_CLIPPED_ALBERTA_REMAPPED.tif"
TIF_PATH = "/beluga/Hackathon15_AlphaEarth_Alberta/Hackathon15_AlphaEarth_Alberta/experiments/BasicUNet_landsat8_filtered_20260220_123132_224x224_v2/classified_scenes/Alberta_2020_L8_Stacked_6Bands_labels_20260223_085040_CLIPPED.tif"

IGNORE_VALUE = -99

CANADA_TO_ALBERTA = {
    1: 0, 2: 1, 5: 2, 6: 3, 8: 4, 10: 5,
    12: 6, 14: 7, 15: 8, 16: 9, 17: 10, 18: 11, 19: 12,
}

print("Loading arrays...")
with rasterio.open(GT_PATH) as src:
    gt = src.read(1).astype(np.int16)
    gt_nodata = src.nodata

with rasterio.open(TIF_PATH) as src:
    tif = src.read(1).astype(np.int16)
    tif_nodata = src.nodata  # 255

print(f"GT  nodata value : {gt_nodata}")
print(f"TIF nodata value : {tif_nodata}")

# ── Masks ─────────────────────────────────────────────────────────────────────
gt_valid  = gt  != IGNORE_VALUE
tif_valid = tif != int(tif_nodata)   # excludes 255

print(f"\nGT  valid pixels : {gt_valid.sum():,}")
print(f"TIF valid pixels : {tif_valid.sum():,}")
print(f"Difference       : {int(gt_valid.sum()) - int(tif_valid.sum()):+,}")

# ── Case 1: GT valid but TIF is 255 (nodata) ─────────────────────────────────
gt_valid_tif_nodata = gt_valid & (tif == int(tif_nodata))
print(f"\nCase 1 — GT valid  BUT TIF=255 (nodata)  : {gt_valid_tif_nodata.sum():,}")
if gt_valid_tif_nodata.sum() > 0:
    vals, cnts = np.unique(gt[gt_valid_tif_nodata], return_counts=True)
    print("  GT class distribution at these pixels:")
    for v, c in zip(vals, cnts):
        print(f"    GT class {v:>3} → {c:>10,} px")

# ── Case 2: TIF valid but GT is IGNORE ────────────────────────────────────────
tif_valid_gt_invalid = tif_valid & ~gt_valid
print(f"\nCase 2 — TIF valid BUT GT=IGNORE         : {tif_valid_gt_invalid.sum():,}")
if tif_valid_gt_invalid.sum() > 0:
    vals, cnts = np.unique(tif[tif_valid_gt_invalid], return_counts=True)
    print("  TIF raw value distribution at these pixels:")
    for v, c in zip(vals, cnts):
        alberta = CANADA_TO_ALBERTA.get(int(v), "UNKNOWN")
        print(f"    TIF raw {v:>3} (Alberta={alberta}) → {c:>10,} px")

# ── Case 3: Both valid BUT TIF value = 0 (not in Canada mapping) ─────────────
both_valid   = gt_valid & tif_valid
tif_zero     = both_valid & (tif == 0)
print(f"\nCase 3 — Both valid AND TIF raw value = 0 : {tif_zero.sum():,}")
print( "  (0 is NOT a valid Canada-wide ID → remapped to IGNORE_VALUE)")
if tif_zero.sum() > 0:
    vals, cnts = np.unique(gt[tif_zero], return_counts=True)
    print("  GT class distribution at these TIF=0 pixels:")
    for v, c in zip(vals, cnts):
        print(f"    GT class {v:>3} → {c:>10,} px")

# ── Case 4: Both valid but TIF has any other unmapped value ──────────────────
valid_canada_ids = set(CANADA_TO_ALBERTA.keys())
unmapped_mask = both_valid.copy()
for cid in valid_canada_ids:
    unmapped_mask &= (tif != cid)   # remove all known valid IDs
# Also remove 0 (already counted above) and 255 (nodata, already excluded)
unmapped_mask &= (tif != 0) & (tif != int(tif_nodata))
print(f"\nCase 4 — Both valid AND TIF has other unmapped value : {unmapped_mask.sum():,}")
if unmapped_mask.sum() > 0:
    vals, cnts = np.unique(tif[unmapped_mask], return_counts=True)
    print("  Unmapped TIF values:")
    for v, c in zip(vals, cnts):
        print(f"    TIF raw {v:>3} → {c:>10,} px")

# ── Summary of what gets excluded from metrics ────────────────────────────────
total_excluded = gt_valid_tif_nodata.sum() + tif_zero.sum() + unmapped_mask.sum()
print(f"\n{'='*60}")
print(f"SUMMARY OF EXCLUDED VALID GT PIXELS")
print(f"{'='*60}")
print(f"  GT valid total              : {gt_valid.sum():,}")
print(f"  Lost because TIF=255 nodata : {gt_valid_tif_nodata.sum():,}  (Case 1)")
print(f"  Lost because TIF=0 unmapped : {tif_zero.sum():,}  (Case 3)")
print(f"  Lost because other unmapped : {unmapped_mask.sum():,}  (Case 4)")
print(f"  Total excluded              : {total_excluded:,}")
print(f"  Expected overlap valid      : {int(gt_valid.sum()) - total_excluded:,}")
print(f"  Actual overlap valid        : {(gt_valid & tif_valid & ~tif_zero & ~unmapped_mask).sum():,}")
print(f"\n  → These exclusions represent {total_excluded/gt_valid.sum()*100:.4f}% of GT valid pixels")
print(f"  → Metrics are computed on {(int(gt_valid.sum())-total_excluded)/gt_valid.sum()*100:.4f}% of Alberta")

Loading arrays...
GT  nodata value : -99.0
TIF nodata value : 255.0

GT  valid pixels : 711,031,930
TIF valid pixels : 1,159,123,680
Difference       : -448,091,750

Case 1 — GT valid  BUT TIF=255 (nodata)  : 0

Case 2 — TIF valid BUT GT=IGNORE         : 448,091,750
  TIF raw value distribution at these pixels:
    TIF raw   0 (Alberta=UNKNOWN) → 448,070,292 px
    TIF raw   1 (Alberta=0) →      1,029 px
    TIF raw   5 (Alberta=2) →          7 px
    TIF raw   6 (Alberta=3) →        275 px
    TIF raw   8 (Alberta=4) →         52 px
    TIF raw  10 (Alberta=5) →        547 px
    TIF raw  14 (Alberta=7) →          4 px
    TIF raw  15 (Alberta=8) →        160 px
    TIF raw  16 (Alberta=9) →      6,488 px
    TIF raw  17 (Alberta=10) →        126 px
    TIF raw  18 (Alberta=11) →     11,399 px
    TIF raw  19 (Alberta=12) →      1,371 px

Case 3 — Both valid AND TIF raw value = 0 : 21,535
  (0 is NOT a valid Canada-wide ID → remapped to IGNORE_VALUE)
  GT class distribution at these T

In [3]:
"""
Print number of samples per class in ground truth and classified TIF.

Three conditions enforced:
  1. Spatial size of GT and classified scene must match.
  2. Non-valid pixels are identified from GT (-99) and the same mask is applied
     to the classified scene — GT is the single source of truth for validity.
  3. Pixel values must be [0-12]. GT is already in [0-12]. The classified TIF
     is in Canada-wide IDs and is remapped to [0-12] via CANADA_TO_ALBERTA.
"""
import numpy as np
import rasterio

GT_PATH  = "/beluga/Hackathon15_AlphaEarth_Alberta/Hackathon15_AlphaEarth_Alberta/GroundTruth_Landsat_Canada/landcover-2020-classification_CLIPPED_ALBERTA_REMAPPED.tif"
TIF_PATH = "/beluga/Hackathon15_AlphaEarth_Alberta/Hackathon15_AlphaEarth_Alberta/experiments/BasicUNet_landsat8_filtered_20260220_123132_224x224_v2/classified_scenes/Alberta_2020_L8_Stacked_6Bands_labels_20260223_085040_CLIPPED.tif"

IGNORE_VALUE = -99

ALBERTA_CLASSES = {
    0:  "Temperate needleleaf forest",
    1:  "Sub-polar taiga forest",
    2:  "Temperate broadleaf forest",
    3:  "Mixed forest",
    4:  "Temperate shrubland",
    5:  "Temperate grassland",
    6:  "Polar grassland-lichen",
    7:  "Wetland",
    8:  "Cropland",
    9:  "Barren lands",
    10: "Urban",
    11: "Water",
    12: "Snow/ice",
}

CANADA_TO_ALBERTA = {
    1: 0, 2: 1, 5: 2, 6: 3, 8: 4, 10: 5,
    12: 6, 14: 7, 15: 8, 16: 9, 17: 10, 18: 11, 19: 12,
}

# =============================================================================
# LOAD
# =============================================================================
print("Loading ground truth...")
with rasterio.open(GT_PATH) as src:
    gt = src.read(1).astype(np.int16)

print("Loading classified TIF...")
with rasterio.open(TIF_PATH) as src:
    tif_raw = src.read(1).astype(np.int16)

# =============================================================================
# CONDITION 1 — Spatial size must match
# =============================================================================
print("\n" + "=" * 60)
print("CONDITION 1: Spatial size check")
print("=" * 60)
print(f"  GT shape  : {gt.shape}")
print(f"  TIF shape : {tif_raw.shape}")

if gt.shape != tif_raw.shape:
    raise ValueError(
        f"Shape mismatch: GT={gt.shape} vs TIF={tif_raw.shape}. "
        "Cannot proceed — arrays must cover the same spatial extent."
    )
print("  ✅ Shapes match")

# =============================================================================
# CONDITION 3 — Remap classified TIF from Canada-wide IDs → Alberta [0-12]
#               (done before applying the mask so remapping is clean)
# =============================================================================
print("\n" + "=" * 60)
print("CONDITION 3: Remap classified TIF to Alberta labels [0-12]")
print("=" * 60)
print(f"  Raw TIF unique values : {np.unique(tif_raw).tolist()}")

# Start with all pixels as IGNORE_VALUE; overwrite only known Canada IDs
tif = np.full(tif_raw.shape, IGNORE_VALUE, dtype=np.int16)
for canada_id, alberta_id in CANADA_TO_ALBERTA.items():
    tif[tif_raw == canada_id] = alberta_id

print(f"  Remapped TIF unique   : {np.unique(tif).tolist()}")
print(f"  (any value not in CANADA_TO_ALBERTA stays as {IGNORE_VALUE})")

# Verify GT is already in [0-12] (plus IGNORE_VALUE=-99)
gt_unique = np.unique(gt).tolist()
print(f"\n  GT unique values : {gt_unique}")
unexpected_gt = [v for v in gt_unique if v != IGNORE_VALUE and (v < 0 or v > 12)]
if unexpected_gt:
    raise ValueError(f"GT contains unexpected values outside [0-12]: {unexpected_gt}")
print("  ✅ GT values confirmed in [0-12] (plus -99 background)")
print("  ✅ TIF remapped to [0-12] (unmapped values set to -99)")

# =============================================================================
# CONDITION 2 — Valid mask comes from GT only; apply to BOTH arrays
# =============================================================================
print("\n" + "=" * 60)
print("CONDITION 2: Build valid mask from GT, apply to both arrays")
print("=" * 60)

# Valid pixel = GT is a known Alberta class [0-12], i.e. not background (-99)
valid_mask  = (gt != IGNORE_VALUE) & (gt >= 0) & (gt <= 12)
total_pixels = gt.size
valid_total  = int(valid_mask.sum())

print(f"  Total pixels          : {total_pixels:,}")
print(f"  Background in GT (-99): {int((gt == IGNORE_VALUE).sum()):,}  ({(gt == IGNORE_VALUE).mean()*100:.2f}%)")
print(f"  Valid pixels (mask)   : {valid_total:,}  ({valid_mask.mean()*100:.2f}%)")
print("  ✅ Same valid mask applied to both GT and classified TIF")

# Apply GT mask to both — guarantees identical spatial coverage
gt_valid  = gt[valid_mask]   # 1-D array of GT  labels under the mask
tif_valid = tif[valid_mask]  # 1-D array of TIF labels under the same mask

# Report pixels that are GT-valid but TIF has no prediction (stayed -99)
tif_unmapped = int((tif_valid == IGNORE_VALUE).sum())
if tif_unmapped > 0:
    print(f"\n  ⚠️  {tif_unmapped:,} pixels valid in GT but unmapped in TIF ({tif_unmapped/valid_total*100:.4f}%)")
    print(f"     These appear in GT counts but show as 0 in Pred counts.")
else:
    print(f"\n  ✅ All GT-valid pixels have a valid prediction in TIF")

# =============================================================================
# PRINT TABLE
# =============================================================================
W = 95
print()
print("=" * W)
print("  CLASS SAMPLE COUNTS  (denominator = GT-valid pixels only)")
print("=" * W)
print(f"  {'Cls':<5} {'Class Name':<35} {'GT Count':>14} {'GT %':>8} {'Pred Count':>14} {'Pred %':>8}")
print("-" * W)

for i in range(13):
    name    = ALBERTA_CLASSES[i]
    gt_n    = int((gt_valid  == i).sum())
    tif_n   = int((tif_valid == i).sum())
    # Both percentages use the same denominator (GT-valid total) for fair comparison
    gt_pct  = gt_n  / valid_total * 100
    tif_pct = tif_n / valid_total * 100
    print(f"  {i:<5} {name:<35} {gt_n:>14,} {gt_pct:>7.3f}% {tif_n:>14,} {tif_pct:>7.3f}%")

print("-" * W)
gt_class_sum  = int(sum((gt_valid  == i).sum() for i in range(13)))
tif_class_sum = int(sum((tif_valid == i).sum() for i in range(13)))
print(f"  {'TOTAL (classes 0-12)':<41} {gt_class_sum:>14,} {gt_class_sum/valid_total*100:>7.3f}% {tif_class_sum:>14,} {tif_class_sum/valid_total*100:>7.3f}%")
if tif_unmapped > 0:
    print(f"  {'TIF unmapped under GT-valid mask':<41} {'':>14}  {'':>8} {tif_unmapped:>14,} {tif_unmapped/valid_total*100:>7.3f}%")
print("=" * W)
print(f"\n  GT-valid pixels (basis for all metrics) : {valid_total:,}")
print(f"  TIF predicted (of GT-valid)             : {tif_class_sum:,}")
print(f"  TIF unmapped  (of GT-valid)             : {tif_unmapped:,}  ({tif_unmapped/valid_total*100:.4f}%)")

Loading ground truth...
Loading classified TIF...

CONDITION 1: Spatial size check
  GT shape  : (44616, 25980)
  TIF shape : (44616, 25980)
  ✅ Shapes match

CONDITION 3: Remap classified TIF to Alberta labels [0-12]
  Raw TIF unique values : [0, 1, 5, 6, 8, 10, 14, 15, 16, 17, 18, 19]
  Remapped TIF unique   : [-99, 0, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12]
  (any value not in CANADA_TO_ALBERTA stays as -99)

  GT unique values : [-99, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  ✅ GT values confirmed in [0-12] (plus -99 background)
  ✅ TIF remapped to [0-12] (unmapped values set to -99)

CONDITION 2: Build valid mask from GT, apply to both arrays
  Total pixels          : 1,159,123,680
  Background in GT (-99): 448,091,750  (38.66%)
  Valid pixels (mask)   : 711,031,930  (61.34%)
  ✅ Same valid mask applied to both GT and classified TIF

  ⚠️  21,535 pixels valid in GT but unmapped in TIF (0.0030%)
     These appear in GT counts but show as 0 in Pred counts.

  CLASS SAMPLE COUNTS  (denomi

LANDSAT8_BAND_GROUPS = {
    "VIS":  [0, 1, 2],   # Blue, Green, Red
    "NIR":  [3],          # Near-infrared
    "SWIR": [4, 5],       # Shortwave infrared
}
```

Instead of treating all 6 bands identically, the stem **splits them by physical meaning**. Each group gets its own `1×1 Conv2d` that projects it to a common `hidden_dim` channel size. This produces 3 streams of shape `(B, hidden_dim, H, W)` — one per group.

---

## 3. The mHC Mechanism — the core idea

The mHC block answers: *"given n streams, how should they influence each other?"* It learns three **per-pixel** mixing matrices from the data itself:
 N = B×H×W tokens
 n: Number of streams
 C: number of bands
| Matrix | Shape | Role |
|--------|-------|------|
| `H_pre` | `(N, 1, n)` | Aggregation weights — blends n streams into one vector fed to Mamba |
| `H_post` | `(N, 1, n)` | Expansion weights — maps Mamba's output back out to n streams |
| `H_res` | `(N, n, n)` | Residual mixing — lets streams interact directly with each other |

These are generated by `LocalMHCGenerator`, a small MLP that takes all streams concatenated as input. Crucially, `H_res` is passed through **Sinkhorn normalisation** (`sinkhorn_ds_batched`) which makes it **doubly stochastic** (rows and columns all sum to 1) — a soft permutation matrix that controls information flow between streams in a principled way.

---

## 4. The 5-Step Forward Pass — `CompleteMHCBlock`

Each block processes the streams in five steps:
```
Step 1 — Aggregate:   x_l (N,n,C) × H_pre (N,1,n)  → X_pre (N,C)
                      Weighted sum of streams → single token per pixel

Step 2 — Mamba:       X_pre (N,C)  → Z (N,C)
                      Sequence modelling across all pixels (spatial context)

Step 3 — Expand:      Z (N,C) × H_post (N,n)  → delta (N,n,C)
                      Broadcast Mamba output back to all n streams

Step 4 — Mix:         H_res (N,n,n) × x_l (N,n,C)  → h_mixed (N,n,C)
                      Direct stream-to-stream mixing (residual path)

Step 5 — Combine:     h_mixed + delta  → output streams
                      Merge the two paths
```

The key insight is that **Steps 1–3 are the "global" path** (all streams funnelled through Mamba) and **Step 4 is the "local" path** (streams mix directly). Adding them together gives both global sequence context and local spectral interaction.

---

## 5. Mamba — `Step2_SpatialMamba`

Mamba is a **selective state-space model** — an efficient alternative to Transformers for long sequences. Here, the entire image is flattened to a sequence of `N = B×H×W` tokens, each of dimension `hidden_dim`. Mamba processes this sequence to capture **spatial dependencies** across the whole image. Multiple Mamba blocks are stacked with residual connections and LayerNorm.

The number of Mamba blocks **increases** with depth (`mamba_blocks_per_stage = [2, 4, 6]`), giving later stages more capacity for complex spatial reasoning.

---

## 6. Full Model — `mHC_MambaHSI_Landsat8`
```
Input (B, 6, H, W)
    ↓  SpectralGroupStem
3 streams (B, hidden_dim, H, W) each
    ↓  CompleteMHCBlock × 3  (with 2, 4, 6 Mamba blocks respectively)
3 streams updated
    ↓  mean across streams → (B, hidden_dim, H, W)
    ↓  Conv1×1 → BN → ReLU → AdaptiveAvgPool → Linear
Logits (B, num_classes)